### Visão Geral:Este código implementa uma das múltiplas formas de RAG multimodelo. Ele extrai e processa texto e imagens de PDFs, utilizando um sistema Retrieval-Augmented Generation multimodal (RAG) para resumir e recuperar conteúdo para responder a perguntas.

### Componentes-chave:- **PyMuPDF**: Para extrair texto e imagens de PDFs.
- ** Modelo Gemini 1.5-flash**: Para resumir imagens e tabelas.
- **Cohere Embeddings**: Para incorporar splits de documentos.
- **Chroma Vectorstore**: Para armazenar e recuperar incorporações de documentos.
- **LangChain**: Para orquestrar o oleoduto de recuperação e geração.

Diagrama:
<img src="../images/multi_model_rag_with_captioning.svg" alt="Reliable-RAG" width="300">

Motivação:
Resumir eficientemente documentos complexos para facilitar a fácil recuperação e respostas concisas para dados multimodais.

### Detalhes do método:- Texto e imagens são extraídos do PDF usando PyMuPDF.
- A somarização é realizada em imagens e tabelas extraídas usando Gemini.
- Embutimentos são gerados via Cohere para armazenamento em Chroma.
- Um retriever baseado em similaridade obtém seções relevantes com base na consulta.

Benefícios:
- Recuperação simplificada de documentos complexos e multimodais.
- Processo Q&A simplificado para texto e imagens.
- Arquitetura flexível para expandir para mais tipos de documentos.

Implementação:
- Os documentos são divididos em chunks com sobreposição usando um divisor de texto.
- Texto resumido e conteúdo de imagem são armazenados como vetores.
- As consultas são tratadas recuperando segmentos de documentos relevantes e gerando respostas concisas.

### Resumo:O projeto permite o processamento e recuperação de documentos multimodais, fornecendo respostas concisas e relevantes, combinando LLMs de última geração e sistemas de recuperação baseados em vetores.

# Instalação e Importações do Pacote
A célula abaixo instala todos os pacotes necessários para executar este notebook.
.

In [ ]:
# Install required packages
!pip install langchain langchain-community pillow pymupdf python-dotenv

In [1]:
import fitz  # PyMuPDF
from PIL import Image
import io
import os
from dotenv import load_dotenv

import google.generativeai as genai
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_cohere import ChatCohere, CohereEmbeddings

load_dotenv()

True

Baixe o papel "Atenção é tudo o que precisa"

In [2]:
!wget https://arxiv.org/pdf/1706.03762
!mv 1706.03762 attention_is_all_you_need.pdf

--2024-09-20 19:19:26--  https://arxiv.org/pdf/1706.03762
Resolving arxiv.org (arxiv.org)... 151.101.195.42, 151.101.3.42, 151.101.67.42, ...
Connecting to arxiv.org (arxiv.org)|151.101.195.42|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2215244 (2.1M) [application/pdf]
Saving to: ‘1706.03762’

1706.03762          100%[===================>]   2.11M  13.3MB/s    in 0.2s    

2024-09-20 19:19:26 (13.3 MB/s) - ‘1706.03762’ saved [2215244/2215244]



### Extração de dados

In [3]:
text_data = []
img_data = []

In [4]:
with fitz.open('attention_is_all_you_need.pdf') as pdf_file:
    # Create a directory to store the images
    if not os.path.exists("extracted_images"):
        os.makedirs("extracted_images")

    # Loop through every page in the PDF
    for page_number in range(len(pdf_file)):
        page = pdf_file[page_number]
        
        # Get the text on page
        text = page.get_text().strip()
        text_data.append({"response": text, "name": page_number+1})
        # Get the list of images on the page
        images = page.get_images(full=True)

        # Loop through all images found on the page
        for image_index, img in enumerate(images, start=0):
            xref = img[0]  # Get the XREF of the image
            base_image = pdf_file.extract_image(xref)  # Extract the image
            image_bytes = base_image["image"]  # Get the image bytes
            image_ext = base_image["ext"]  # Get the image extension
            
            # Load the image using PIL and save it
            image = Image.open(io.BytesIO(image_bytes))
            image.save(f"extracted_images/image_{page_number+1}_{image_index+1}.{image_ext}")

In [5]:
genai.configure(api_key=os.getenv('GOOGLE_API_KEY'))
model = genai.GenerativeModel(model_name="gemini-1.5-flash")

Legendagem da Imagem

In [6]:
for img in os.listdir("extracted_images"):
    image = Image.open(f"extracted_images/{img}")
    response = model.generate_content([image, "You are an assistant tasked with summarizing tables, images and text for retrieval. \
    These summaries will be embedded and used to retrieve the raw text or table elements \
    Give a concise summary of the table or text that is well optimized for retrieval. Table or text or image:"])
    img_data.append({"response": response.text, "name": img})

Vectostore

In [7]:
# Set embeddings
embedding_model = CohereEmbeddings(model="embed-english-v3.0")

# Load the document
docs_list = [Document(page_content=text['response'], metadata={"name": text['name']}) for text in text_data]
img_list = [Document(page_content=img['response'], metadata={"name": img['name']}) for img in img_data]

# Split
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=400, chunk_overlap=50
)

doc_splits = text_splitter.split_documents(docs_list)
img_splits = text_splitter.split_documents(img_list)

In [8]:
# Add to vectorstore
vectorstore = Chroma.from_documents(
    documents=doc_splits + img_splits, # adding the both text and image splits
    collection_name="multi_model_rag",
    embedding=embedding_model,
)

retriever = vectorstore.as_retriever(
                search_type="similarity",
                search_kwargs={'k': 1}, # number of documents to retrieve
            )

### Consulta

In [9]:
query = "What is the BLEU score of the Transformer (base model)?"

In [10]:
docs = retriever.invoke(query)

Saída

In [11]:
from langchain_core.output_parsers import StrOutputParser

# Prompt
system = """You are an assistant for question-answering tasks. Answer the question based upon your knowledge. 
Use three-to-five sentences maximum and keep the answer concise."""
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Retrieved documents: \n\n <docs>{documents}</docs> \n\n User question: <question>{question}</question>"),
    ]
)

# LLM
llm = ChatCohere(model="command-r-plus", temperature=0)

# Chain
rag_chain = prompt | llm | StrOutputParser()

# Run
generation = rag_chain.invoke({"documents":docs[0].page_content, "question": query})
print(generation)

The Transformer (base model) achieves a BLEU score of 27.3.


![](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=all-rag-techniques--multi-model-rag-with-captioning)